In [10]:
from pathlib import Path
import pandas as pd

In [11]:
project_root = Path.cwd().parent
data_path = project_root / 'data' / 'raw' / 'online_retail_II.xlsx'
assert data_path.exists(), f"Workbook not found: {data_path}"

In [12]:
workbook = pd.ExcelFile(data_path)
print(workbook.sheet_names)

['Year 2009-2010', 'Year 2010-2011']


In [13]:
for sheet_name in workbook.sheet_names:
    sample = pd.read_excel(workbook, sheet_name=sheet_name, nrows=5)

    print(f'Sheet Name: {sheet_name}')
    print(f'Columns: {sample.columns.tolist()}')
    print('\n')
    

Sheet Name: Year 2009-2010
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


Sheet Name: Year 2010-2011
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']




In [14]:
for sheet_name in workbook.sheet_names:
    frame = pd.read_excel(workbook, sheet_name)

    print(f'Sheet Name: {sheet_name}')
    print(f'Shape: {frame.shape}')

    display(pd.DataFrame({
        'dtype': frame.dtypes.astype(str),
        'missing_rows': frame.isna().sum(),
        'missing_percent': frame.isna().mean().mul(100).round(2)
    }))

Sheet Name: Year 2009-2010
Shape: (525461, 8)


,dtype,missing_rows,missing_percent
Invoice,object,0,0.00
StockCode,object,0,0.00
Description,object,2928,0.56
Quantity,int64,0,0.00
InvoiceDate,datetime64[us],0,0.00
Price,float64,0,0.00
Customer ID,float64,107927,20.54
Country,str,0,0.00


Sheet Name: Year 2010-2011
Shape: (541910, 8)


,dtype,missing_rows,missing_percent
Invoice,object,0,0.00
StockCode,object,0,0.00
Description,object,1454,0.27
Quantity,int64,0,0.00
InvoiceDate,datetime64[us],0,0.00
Price,float64,0,0.00
Customer ID,float64,135080,24.93
Country,str,0,0.00


In [15]:
for sheet_name in workbook.sheet_names:
    frame = pd.read_excel(workbook, sheet_name)

    print(f"\nSheet: {sheet_name}")

    print(frame.columns.tolist())
    display(frame.dtypes)
    
    display(frame.isna().sum())
    
    display(frame.describe())
    
    dates = pd.to_datetime(frame["InvoiceDate"], errors="coerce")
    print("Earliest date:", dates.min())
    print("Latest date:", dates.max())
    
    print("Negative quantities:", (frame["Quantity"] < 0).sum())
    print("Zero prices:", (frame["Price"] == 0).sum())
    
    cancellations = (
        frame["Invoice"]
        .astype("string")
        .str.upper()
        .str.startswith("C", na=False)
    )
    print("Cancellation lines:", cancellations.sum())
    
    display(frame.groupby("Invoice").size().sort_values(ascending=False).head())


Sheet: Year 2009-2010
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object

Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107927
Country             0
dtype: int64

,Quantity,InvoiceDate,Price,Customer ID
count,525461.000000,525461,525461.000000,417534.000000
mean,10.337667,2010-06-28 11:37:36.845018,4.688834,15360.645478
min,-9600.000000,2009-12-01 07:45:00,-53594.360000,12346.000000
25%,1.000000,2010-03-21 12:20:00,1.250000,13983.000000
50%,3.000000,2010-07-06 09:51:00,2.100000,15311.000000
75%,10.000000,2010-10-15 12:45:00,4.210000,16799.000000
max,19152.000000,2010-12-09 20:01:00,25111.090000,18287.000000
std,107.424110,NaN,146.126914,1680.811316


Earliest date: 2009-12-01 07:45:00
Latest date: 2010-12-09 20:01:00
Negative quantities: 12326
Zero prices: 3687
Cancellation lines: 10206


...

Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object

Invoice             0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
Price               0
Customer ID    135080
Country             0
dtype: int64

,Quantity,InvoiceDate,Price,Customer ID
count,541910.000000,541910,541910.000000,406830.000000
mean,9.552234,2011-07-04 13:35:22.342307,4.611138,15287.684160
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.080957,NaN,96.759765,1713.603074


Earliest date: 2010-12-01 08:26:00
Latest date: 2011-12-09 12:50:00
Negative quantities: 10624
Zero prices: 2515
Cancellation lines: 9288


Invoice
573585    1114
581219     749
581492     731
580729     721
558475     705
dtype: int64

## Cross-Sheet overlap investigation

In [16]:
sheets = pd.read_excel(data_path, sheet_name=None)

In [17]:
parts = []

for sheet_name, frame in sheets.items():
    part = frame.copy()
    part['source_sheet'] = sheet_name
    parts.append(part)

combined = pd.concat(parts, ignore_index=True)
print(f"Combined rows: {len(combined)}")

Combined rows: 1067371


In [18]:
business_columns = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country",
]

distinct_per_sheet = combined.drop_duplicates(
    subset = business_columns+['source_sheet']
)

cross_sheet_matches = distinct_per_sheet[
    distinct_per_sheet.duplicated(
        subset=business_columns,
        keep=False
    )
]

print(
    "Distinct business rows appearing in multiple sheets:",
    len(cross_sheet_matches.drop_duplicates(subset=business_columns)),
)

display(
    cross_sheet_matches
    .sort_values(["InvoiceDate", "Invoice", "StockCode"])
    .head(20)
)

Distinct business rows appearing in multiple sheets: 22202


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet
502944,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom,Year 2009-2010
525467,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom,Year 2010-2011
502943,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom,Year 2009-2010
525466,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom,Year 2010-2011
502939,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,Year 2009-2010
525462,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,Year 2010-2011
502942,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,Year 2009-2010
525465,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,Year 2010-2011
502941,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,Year 2009-2010
525464,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,Year 2010-2011


In [19]:
is_cancellation = (
    combined["Invoice"]
    .astype("string")
    .str.upper()
    .str.startswith("C", na=False)
)

negative_without_c = combined[
    (combined["Quantity"] < 0) & ~is_cancellation
]

print("Negative quantities without C prefix:", len(negative_without_c))

display(
    negative_without_c[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "Price",
            "Customer ID",
            "source_sheet",
        ]
    ].head(20)
)

Negative quantities without C prefix: 3457


,Invoice,StockCode,Description,Quantity,Price,Customer ID,source_sheet
263,489464,21733,85123a mixed,-96,0.0,NaN,Year 2009-2010
283,489463,71477,short,-240,0.0,NaN,Year 2009-2010
284,489467,85123A,21733 mixed,-192,0.0,NaN,Year 2009-2010
470,489521,21646,NaN,-50,0.0,NaN,Year 2009-2010
3114,489655,20683,NaN,-44,0.0,NaN,Year 2009-2010
3162,489660,35956,lost,-1043,0.0,NaN,Year 2009-2010
3168,489663,35605A,damages,-117,0.0,NaN,Year 2009-2010
4296,489806,18010,NaN,-770,0.0,NaN,Year 2009-2010
4538,489820,21133,invcd as 84879?,-720,0.0,NaN,Year 2009-2010
4566,489821,85049G,NaN,-240,0.0,NaN,Year 2009-2010


In [20]:
adjustment_summary = pd.Series({
    "total_rows": len(negative_without_c),
    "zero_price_rows": negative_without_c["Price"].eq(0).sum(),
    "missing_customer_rows": negative_without_c["Customer ID"].isna().sum(),
    "zero_price_and_missing_customer_rows": (
        negative_without_c["Price"].eq(0)
        & negative_without_c["Customer ID"].isna()
    ).sum(),
})

display(adjustment_summary)

total_rows                              3457
zero_price_rows                         3457
missing_customer_rows                   3457
zero_price_and_missing_customer_rows    3457
dtype: int64

In [21]:
typical_adjustment = (
    negative_without_c["Price"].eq(0)
    & negative_without_c["Customer ID"].isna()
)

display(
    negative_without_c.loc[~typical_adjustment].head(20)
)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet


### Cross-sheet overlap findings

- The sheets overlap in December 2010.
- 22,202 distinct business rows appear in both sheets.
- Sample matches have identical values across all eight business columns.
- Combining the sheets without handling this overlap could inflate metrics.
- Raw records are preserved while we investigate a cleaning rule.

### Negative quantities without cancellation prefixes

Found 3,457 rows with negative quantities whose invoices do not
begin with C. All have zero prices and missing customer IDs.

Descriptions such as "lost", "damages", and "short" suggest stock
adjustments or administrative corrections.

Decision: preserve these records in the raw data, but exclude
them from customer purchase activity.

These counts include any overlap between the source sheets.

In [22]:
from sqlalchemy import create_engine, text

engine = create_engine(
    "postgresql+psycopg://husein@localhost:5432/revenueiq"
)

with engine.connect() as connection:
    result = connection.execute(
        text("SELECT current_database(), current_user")
    )
    print(result.one())

('revenueiq', 'husein')


## Test ingestion: 10 source rows

In [25]:
sheet_name = "Year 2009-2010"

sample = pd.read_excel(
    data_path,
    sheet_name=sheet_name,
    nrows=10,
    dtype={
        "Invoice": "string",
        "StockCode": "string",
        "Customer ID": "string",
    },
)

sample = sample.rename(columns={
    "Invoice": "invoice",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "Price": "unit_price",
    "Customer ID": "customer_id",
    "Country": "country",
})

sample["source_file"] = data_path.name
sample["source_sheet"] = sheet_name
sample["source_row_number"] = range(2, len(sample) + 2)

display(sample)

,invoice,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,source_file,source_sheet,source_row_number
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,online_retail_II.xlsx,Year 2009-2010,2
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,online_retail_II.xlsx,Year 2009-2010,3
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,online_retail_II.xlsx,Year 2009-2010,4
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,online_retail_II.xlsx,Year 2009-2010,5
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,online_retail_II.xlsx,Year 2009-2010,6
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085,United Kingdom,online_retail_II.xlsx,Year 2009-2010,7
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,online_retail_II.xlsx,Year 2009-2010,8
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085,United Kingdom,online_retail_II.xlsx,Year 2009-2010,9
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085,United Kingdom,online_retail_II.xlsx,Year 2009-2010,10
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085,United Kingdom,online_retail_II.xlsx,Year 2009-2010,11


In [26]:
from decimal import Decimal

records = []

for row in sample.to_dict(orient="records"):
    record = {}

    for column, value in row.items():
        if pd.isna(value):
            record[column] = None
        elif column == "unit_price":
            record[column] = Decimal(str(value))
        elif column == "invoice_date":
            record[column] = pd.Timestamp(value).to_pydatetime()
        else:
            record[column] = value

    records.append(record)

In [29]:
insert_sql = text("""
    INSERT INTO raw.retail_transactions (
        source_file,
        source_sheet,
        source_row_number,
        invoice,
        stock_code,
        description,
        quantity,
        invoice_date,
        unit_price,
        customer_id,
        country
    )
    VALUES (
        :source_file,
        :source_sheet,
        :source_row_number,
        :invoice,
        :stock_code,
        :description,
        :quantity,
        :invoice_date,
        :unit_price,
        :customer_id,
        :country
    )
    ON CONFLICT ON CONSTRAINT unique_source_row DO NOTHING
""")

with engine.begin() as connection:
    connection.execute(insert_sql, records)

print("Sample import completed.")

Sample import completed.


In [30]:
with engine.connect() as connection:
    total = connection.execute(
        text("SELECT COUNT(*) FROM raw.retail_transactions")
    ).scalar_one()

print("Rows in raw table:", total)

Rows in raw table: 10


In [31]:
def prepare_sheet(frame, source_file, sheet_name):
    frame = frame.rename(columns={
        "Invoice": "invoice",
        "StockCode": "stock_code",
        "Description": "description",
        "Quantity": "quantity",
        "InvoiceDate": "invoice_date",
        "Price": "unit_price",
        "Customer ID": "customer_id",
        "Country": "country",
    }).copy()

    frame["source_file"] = source_file
    frame["source_sheet"] = sheet_name
    frame["source_row_number"] = range(2, len(frame) + 2)

    return frame

In [32]:
def import_sheet(engine, frame, batch_size=5000):
    for start in range(0, len(frame), batch_size):
        batch = frame.iloc[start:start + batch_size]
        records = []

        for row in batch.to_dict(orient="records"):
            record = {}

            for column, value in row.items():
                if pd.isna(value):
                    record[column] = None
                elif column == "unit_price":
                    record[column] = Decimal(str(value))
                elif column == "invoice_date":
                    record[column] = pd.Timestamp(value).to_pydatetime()
                else:
                    record[column] = value

            records.append(record)

        with engine.begin() as connection:
            connection.execute(insert_sql, records)

        if start == 0 or (start // batch_size + 1) % 20 == 0:
            print(f"Processed {min(start + batch_size, len(frame)):,} rows")

    print(f"Finished processing {len(frame):,} source rows.")

In [34]:
with pd.ExcelFile(data_path) as workbook:
    for sheet_name in workbook.sheet_names:
        print(f"\nReading {sheet_name}...")

        frame = pd.read_excel(
            workbook,
            sheet_name=sheet_name,
            dtype={
                "Invoice": "string",
                "StockCode": "string",
                "Customer ID": "string",
            },
        )

        prepared = prepare_sheet(
            frame,
            source_file=data_path.name,
            sheet_name=sheet_name,
        )

        import_sheet(engine, prepared)


Reading Year 2009-2010...
Processed 5,000 rows
Processed 100,000 rows
Processed 200,000 rows
Processed 300,000 rows
Processed 400,000 rows
Processed 500,000 rows
Finished processing 525,461 source rows.

Reading Year 2010-2011...
Processed 5,000 rows
Processed 100,000 rows
Processed 200,000 rows
Processed 300,000 rows
Processed 400,000 rows
Processed 500,000 rows
Finished processing 541,910 source rows.


In [35]:
with engine.connect() as connection:
    counts = pd.read_sql(
        text("""
            SELECT
                source_sheet,
                COUNT(*) AS row_count
            FROM raw.retail_transactions
            GROUP BY source_sheet
            ORDER BY source_sheet
        """),
        connection,
    )

display(counts)
print("Total rows:", counts["row_count"].sum())

,source_sheet,row_count
0,Year 2009-2010,525461
1,Year 2010-2011,541910


Total rows: 1067371


### Order timestamp convention

64 invoices contain multiple line timestamps. All are within
the same calendar day: 63 span one minute and one spans two minutes.
Customer and country are consistent within each eligible invoice.

We treat each invoice as one order and use its earliest eligible
line timestamp as order_date. We retain the latest line timestamp
as last_line_date. The cause of the timestamp variation is unknown.